In [1]:
# Data manipulation
import pandas as pd
import numpy as np 

# Visualization
import seaborn as sns 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ML
# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline #for preprocessing steps

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import collections

# Model selection
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV

# Metrics
from sklearn.metrics import (recall_score, precision_score, f1_score, classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_recall_curve, 
                             average_precision_score)

# Dealing with imbalanced dataset
from imblearn.pipeline import make_pipeline
from imblearn.metrics import classification_report_imbalanced
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler, NearMiss




In [3]:
features = pd.read_csv('raw_features.csv')
features = features.set_index('ID')

In [4]:
cat_cols = ['KYC', 'TERMS_VERSION']

X_raw = features.drop(columns=['IS_FRAUDSTER', 'RECORDED'])  # RECORDED was constant/leaky, see EDA above
y_raw = features['IS_FRAUDSTER'].astype(int)

numeric_cols = [c for c in X_raw.columns if c not in cat_cols]

In [6]:
# We will oversample during cross validating
# Split on train/test 
X_temp_over, X_test_over, y_temp_over, y_test_over = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)
# put aside 'test' for real test and split 'train' once more on train/test for undersampling crossvalidation. 
X_train_under, X_val_under, y_train_under, y_val_under = train_test_split(
    X_temp_over,
    y_temp_over,
    test_size=0.25,
    random_state=42,
    stratify=y_temp_over
)



In [7]:
classifiers = {
    "LogisiticRegression": LogisticRegression(max_iter=1000, solver='liblinear'), #solver='newton-cholesky', changed from lbfgs, small n_samples >> n_features)
    "KNearest": KNeighborsClassifier(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=100, max_depth=5)
}
# 

In [8]:
numeric_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', RobustScaler()),
])

preprocess = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
for name, clf in classifiers.items():
    pipe = ImbPipeline([
        ('preprocess', preprocess),            # ColumnTransformer- impute+scale numeric, one-hot cat
        ('oversample', SMOTE(random_state=42)),
        ('clf', clf),
    ])

    scores = cross_validate(pipe, X_train_under, y_train_under, cv=cv, scoring=['roc_auc', 'precision', 'recall', 'f1'])
    
    roc_auc = np.round(scores['test_roc_auc'].mean() * 100, 2)
    precision = np.round(scores['test_precision'].mean() * 100, 2)
    recall = np.round(scores['test_recall'].mean() * 100, 2)
    f1 = np.round(scores['test_f1'].mean() * 100, 2)

    print(f"Classifier - {name}: roc_auc={roc_auc}%, recall={recall}%, precision={precision}%, f1-score={f1}%")